In [1]:
import os, fcntl, time
import numpy as np
from pynq import Overlay, allocate
from rtlsdr import RtlSdr

print('[1] Loading Overlay...')
ol = Overlay('./base.bit')
pAudio = ol.audio_codec_ctrl_0
dma = ol.rtl_sdr.axi_dma_0

print('[2] Configuring Audio Codec (ADAU1761)...')
pAudio.configure(sample_rate=48000, iic_index=1, uio_name='audio-codec-ctrl')

# Unmute Mixer3/4 and enable Headphone Out via I2C directly
I2C_SLAVE = 0x0703
fd = os.open('/dev/i2c-1', os.O_RDWR)
fcntl.ioctl(fd, I2C_SLAVE, 0x3B)
def write_reg(addr, val):
    os.write(fd, bytes([0x40, addr, val]))

write_reg(0x1C, 0x21)  # R22 Playback Mixer Left Control 0
write_reg(0x1E, 0x41)  # R24 Playback Mixer Right Control 0
vol = (60 << 2) | 0x03 # Volume 60/63 + Headphone Enable
write_reg(0x23, vol)   # R29 Headphone Left Volume
write_reg(0x24, vol)   # R30 Headphone Right Volume
os.close(fd)
print('    ADAU1761 Mixer & Headphone UNMUTED!')


[1] Loading Overlay...


/usr/local/share/pynq-venv/lib/python3.8/site-packages/pynq/bitstream.py:151: UserWarning: The provided name './base.bit' resulted in multiple possible matches:
 - /home/xilinx/jupyter_notebooks/audio/test/base.bit
 - /usr/local/share/pynq-venv/lib/python3.8/site-packages/pynq/overlays/./base/./base.bit
The first entry of this list, '/home/xilinx/jupyter_notebooks/audio/test/base.bit', will be used, please provide the full path in case your target file was a different one in this list.
  warnings.warn(msg, UserWarning)


[2] Configuring Audio Codec (ADAU1761)...
    ADAU1761 Mixer & Headphone UNMUTED!


/usr/local/share/pynq-venv/lib/python3.8/site-packages/pynq/overlay.py:681: UserWarning: Interrupt mm2s_introut not created: Could not find UIO device for interrupt pin for IRQ number 64
  warnings.warn('Interrupt {} not created: {}'.format(
/usr/local/share/pynq-venv/lib/python3.8/site-packages/pynq/overlay.py:681: UserWarning: Interrupt s2mm_introut not created: Could not find UIO device for interrupt pin for IRQ number 65
  warnings.warn('Interrupt {} not created: {}'.format(


In [2]:
# Step A: 2-second test tone
print('[3] Testing 440 Hz Tone from FPGA hardware for 2 seconds...')
pAudio.mmio.write(0x20, 1)
time.sleep(2.0)
pAudio.mmio.write(0x20, 0)
print('    Tone finished!')

[3] Testing 440 Hz Tone from FPGA hardware for 2 seconds...
    Tone finished!


In [3]:
# check IP contents in bitstream
print(ol.ip_dict.keys())

dma = ol.rtl_sdr.axi_dma_0

dict_keys(['rtl_sdr/axi_dma_0', 'switches_gpio', 'btns_gpio', 'rgbleds_gpio', 'leds_gpio', 'audio_codec_ctrl_0', 'video/axi_vdma_0', 'video/axi_dynclk_0', 'video/v_tc_0', 'simple_io_0', 'axi_quad_spi_0', 'lcd_gpio', 'ps7_0'])


In [4]:
len_in = 1024 * 1500
len_out = int(len_in / 40)

resize = 1 / 40

In [5]:
# make input buffer and output buffer for AXI DMA
input_buffer = allocate(shape=(len_in,), dtype=np.uint16)
output_buffer = allocate(shape=(len_out,), dtype=np.int16)

print(f"Input Buffer Address: {hex(input_buffer.device_address)}")
print(f"Output Buffer Address: {hex(output_buffer.device_address)}")

Input Buffer Address: 0x16900000
Output Buffer Address: 0x16860000


In [6]:
# connect and configure RTL-SDR
sdr = RtlSdr()

fc = 99750000
fs = 1920000
ppm = 60

sdr.sample_rate = fs
sdr.center_freq = fc
sdr.freq_correction = ppm
sdr.gain = "auto"

In [7]:
# increase usb memory buffer
!echo 0 > /sys/module/usbcore/parameters/usbfs_memory_mb

In [8]:
from IPython.display import Audio
import nest_asyncio
import asyncio

In [9]:
nest_asyncio.apply()

flag = 0

it_num = 10

async def streaming():
    it = 0
    async for data in sdr.stream(num_samples_or_bytes=len_in*2, format='bytes', loop=None):
        global flag
        
        raw_uint8 = np.frombuffer(data, dtype=np.uint8)
        
        raw_int8 = (raw_uint8.astype(np.int16) - 128).astype(np.int8)
        
        packed_samples = raw_int8.view(np.uint16)
        
        np.copyto(input_buffer, packed_samples)
        
        if flag == 1:
            dma.sendchannel.wait()
            dma.recvchannel.wait()
        else:
            flag = 1
            
        dma.sendchannel.transfer(input_buffer)
        dma.recvchannel.transfer(output_buffer)
        
#         audio_data = output_buffer.astype(np.float32) / 32768.0
        
#         display(Audio(audio_data, autoplay=True, rate=48000, normalize=False))
        
        it = it + 1
        print("sample %d" % it)
        if it == it_num:
            print("Stopping stream...")
            await sdr.stop()
            sdr.close()
            break
        
asyncio.run(streaming())

sample 1
sample 2
sample 3
sample 4
sample 5
sample 6
sample 7
sample 8
sample 9
sample 10
Stopping stream...


In [10]:
# delete input buffer and output buffer
del input_buffer, output_buffer